# Epic: Did that lemma come from speech or narrative?

THIS FILE USES THE OUT-DATED STANZA-BASED LEMMATIZATION. USE `epic_speech_vs_narrative2.ipynb` instead. (Except for things not needing lemmatization, like the comparison of line counts.)

Given a lemma, what is the likelihood that it came from direct speech versus narrative?

In [12]:
import numpy as np
import pandas as pd

# narrative_df = pd.read_parquet("../parquet/homeric_narrative_dtm.parquet")
# speeches_df = pd.read_parquet("../parquet/homeric_speeches_dtm.parquet")

In [13]:
# n_lemmata_iliad_narrative = narrative_df["Iliad"].sum()
# n_lemmata_odyssey_narrative = narrative_df["Odyssey"].sum()

# print(f"Number of (non-stopword) lemmata in Iliadic narrative: {n_lemmata_iliad_narrative}")
# print(f"Number of (non-stopword) lemmata in Odyssean narrative: {n_lemmata_odyssey_narrative}")
# print("\n\n")

# iliad_narrative_pcts = narrative_df["Iliad"] / n_lemmata_iliad_narrative
# odyssey_narrative_pcts = narrative_df["Odyssey"] / n_lemmata_odyssey_narrative

# print(iliad_narrative_pcts.sort_values(ascending=False).head(100))
# iliad_narrative_pcts.sort_values(ascending=False).head(100).to_csv("../csv/iliad_narrative_top_100_lemmata.csv")
# print("\n\n")
# print(odyssey_narrative_pcts.sort_values(ascending=False).head(100))
# odyssey_narrative_pcts.sort_values(ascending=False).head(100).to_csv("../csv/odyssey_narrative_top_100_lemmata.csv")

# print("\n\n")

# total_lemmata_narrative = n_lemmata_iliad_narrative + n_lemmata_odyssey_narrative
# narrative_df["all"] = narrative_df["Iliad"] + narrative_df["Odyssey"]
# all_pcts = narrative_df["all"] / total_lemmata_narrative

# all_pcts.sort_values(ascending=False).head(100).to_csv("../csv/all_narrative_top_100_lemmata.csv")


In [14]:
# n_lemmata_speeches = speeches_df.sum(axis=1).sum()
# speeches_pcts = speeches_df.sum(axis=1) / n_lemmata_speeches

# speeches_pcts.sort_values(ascending=False).head(100).to_csv("../csv/all_speeches_top_100_lemmata.csv")

In [15]:
# n_lemmata_iliad_speeches = speeches_df["Iliad"].sum(axis=1).sum()
# n_lemmata_odyssey_speeches = speeches_df["Odyssey"].sum(axis=1).sum()

# iliad_speeches_pcts = speeches_df["Iliad"].sum(axis=1) / n_lemmata_iliad_speeches
# odyssey_speeches_pcts = speeches_df["Odyssey"].sum(axis=1) / n_lemmata_odyssey_speeches

# iliad_speeches_pcts.sort_values(ascending=False).head(100).to_csv("../csv/iliad_speeches_top_100_lemmata.csv")
# odyssey_speeches_pcts.sort_values(ascending=False).head(100).to_csv("../csv/odyssey_speeches_top_100_lemmata.csv")


## Speech versus narrative 

In [16]:
df = pd.read_parquet("../parquet/homer_speech_narrative.parquet")

def build_totals(df: pd.DataFrame) -> dict[tuple[str, str], int]:
    return {key: int(df[key].sum()) for key in df.columns}

totals = build_totals(df)

In [17]:
narrative_dtm = df.iliad["narrative"] + df.odyssey["narrative"]
speech_dtm = df.iliad["speech"] + df.odyssey["speech"]

In [18]:
narrative_dtm["δίκαιος"]

np.int64(2)

In [19]:
def log_likelihood(a: pd.Series, b: pd.Series, total_a: int, total_b: int) -> pd.Series:
    """Dunning G² for each lemma. Positive = skewed toward corpus A."""
    N = total_a + total_b
    expected_a = (a + b) * total_a / N
    expected_b = (a + b) * total_b / N

    def safe_term(obs, exp):
        return np.where(obs > 0, obs * np.log(obs / exp), 0)

    g2 = 2 * (safe_term(a, expected_a) + safe_term(b, expected_b))
    return pd.Series(np.where(a / total_a >= b / total_b, g2, -g2), index=a.index)

In [20]:
totals_iliad = totals["iliad", "speech"] + totals["iliad", "narrative"]
totals_odyssey = totals["odyssey", "speech"] + totals["odyssey", "narrative"]

In [21]:
il = df["iliad"].sum(axis=1)
od = df["odyssey"].sum(axis=1)

loglikelihood_work = log_likelihood(il, od, totals_iliad, totals_odyssey)

/Users/pletcher/code/writing/articles/2026-06-13_ccc-2026/.venv/lib/python3.13/site-packages/pandas/core/arraylike.py:402: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)


In [22]:
loglikelihood_work.sort_values(ascending=False).to_csv("../csv/iliad_vs_odyssey.csv")

In [23]:
import altair as alt

In [24]:
speech = df.xs("speech", axis=1, level="register").sum(axis=1)
narrative = df.xs("narrative", axis=1, level="register").sum(axis=1)

totals_speech = totals["iliad", "speech"] + totals["odyssey", "speech"]
totals_narrative = totals["iliad", "narrative"] + totals["odyssey", "narrative"]

loglikelihood_register = log_likelihood(speech, narrative, totals_speech, totals_narrative)

/Users/pletcher/code/writing/articles/2026-06-13_ccc-2026/.venv/lib/python3.13/site-packages/pandas/core/arraylike.py:402: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)


In [25]:
loglikelihood_register.sort_values(ascending=False).to_csv("../csv/speech_vs_narrative.csv")

## Plotting lemmata by work

In [26]:
from scipy.stats import chi2
from statsmodels.stats.multitest import multipletests

total_tokens = totals_iliad + totals_odyssey
overall_freq = (il + od) / total_tokens

work_plot_df = pd.DataFrame({
    "lemma": loglikelihood_work.index,
    "log_likelihood": loglikelihood_work.values,
    "relative_frequency": overall_freq.values,
    "work": np.where(loglikelihood_work.values >= 0, "Iliad", "Odyssey"),
})

# p-value from χ^2(df=1); use abs() because G^2 is signed here
work_plot_df["p_value"] = chi2.sf(work_plot_df["log_likelihood"].abs(), df=1)

# Benjamini-Hochberg FDR correction across all lemmata
_, work_plot_df["p_value_fdr"], _, _ = multipletests(work_plot_df["p_value"], method="fdr_bh")

work_plot_df = work_plot_df[work_plot_df["log_likelihood"].abs() >= 10]
print(f"{len(work_plot_df)} lemmata after filtering")
work_plot_df.sort_values("log_likelihood", ascending=False).head(10)

553 lemmata after filtering


,lemma,log_likelihood,relative_frequency,work,p_value,p_value_fdr
7412,Ἕκτωρ,487.773679,0.004148,Iliad,4.347468e-108,1.828762e-104
863,Τρώς,454.386252,0.005940,Iliad,8.008588e-101,2.245875e-97
6422,Ἀχιλλεύς,299.521223,0.003594,Iliad,4.188718e-67,5.873280e-64
7677,ἵππος,251.333443,0.004307,Iliad,1.329659e-56,1.398302e-53
6420,Ἀχαιός,221.084902,0.006747,Iliad,5.244950e-50,4.902863e-47
34,Αἴας,158.431398,0.001877,Iliad,2.491031e-36,1.905186e-33
4151,πόλεμος,147.011943,0.002618,Iliad,7.800464e-34,5.468775e-31
591,Πάτροκλος,137.398868,0.001464,Iliad,9.864115e-32,5.532453e-29
2844,μάχομαι,133.670569,0.002318,Iliad,6.449656e-31,3.391310e-28
2843,μάχη,120.985764,0.001257,Iliad,3.848839e-28,1.798905e-25


In [27]:
alt.Chart(work_plot_df).mark_point(opacity=0.5, size=30).encode(
    x=alt.X("log_likelihood:Q", title="Log-likelihood (← Odyssey | Iliad →)"),
    y=alt.Y(
        "relative_frequency:Q",
        title="Overall relative frequency",
        scale=alt.Scale(type="log"),
    ),
    color=alt.Color(
        "work:N",
        scale=alt.Scale(domain=["Iliad", "Odyssey"], range=["tomato", "steelblue"]),
        legend=alt.Legend(title="More common in"),
    ),
    tooltip=["lemma:N", "log_likelihood:Q", "relative_frequency:Q"],
).properties(
    title="Iliad vs. Odyssey: lemma distinctiveness vs. overall frequency",
    width=650,
    height=450,
)

alt.Chart(...)

## Plotting lemmata by register

In [28]:
total_tokens = totals_narrative + totals_speech
overall_freq = (narrative + speech) / total_tokens

register_plot_df = pd.DataFrame({
    "lemma": loglikelihood_register.index,
    "log_likelihood": loglikelihood_register.values,
    "relative_frequency": overall_freq.values,
    "register": np.where(loglikelihood_register.values >= 0, "Speech", "Narrative"),
})

# p-value from χ^2(df=1); use abs() because G^2 is signed here
register_plot_df["p_value"] = chi2.sf(register_plot_df["log_likelihood"].abs(), df=1)

# Benjamini-Hochberg FDR correction across all lemmata
_, register_plot_df["p_value_fdr"], _, _ = multipletests(register_plot_df["p_value"], method="fdr_bh")

register_plot_df = register_plot_df[register_plot_df["log_likelihood"].abs() >= 10]
print(f"{len(register_plot_df)} lemmata after filtering")
register_plot_df.sort_values("log_likelihood", ascending=False).head(10)

625 lemmata after filtering


,lemma,log_likelihood,relative_frequency,register,p_value,p_value_fdr
2148,κακός,208.098314,0.004110,Speech,3.570825e-47,1.502068e-43
1981,θεός,159.342964,0.007217,Speech,1.574724e-36,2.604345e-33
3291,ξένος,146.823707,0.001999,Speech,8.575707e-34,1.030678e-30
6499,ἐθέλω,143.006422,0.002750,Speech,5.859044e-33,6.161517e-30
3358,οἶδα,131.164320,0.002853,Speech,2.279354e-30,2.130689e-27
1348,δίδωμι,123.963181,0.004542,Speech,8.582141e-29,7.220155e-26
3355,οἴομαι,117.200384,0.001192,Speech,2.594812e-27,1.984559e-24
6124,ἄνθρωπος,99.165342,0.001783,Speech,2.322766e-23,1.221339e-20
4755,φίλος,91.854373,0.006372,Speech,9.329548e-22,3.924474e-19
4566,τελέω,75.750597,0.001267,Speech,3.218537e-18,1.177285e-15


In [29]:
alt.Chart(register_plot_df).mark_point(opacity=0.5, size=30).encode(
    x=alt.X("log_likelihood:Q", title="Log-likelihood (← Narrative | Speech →)"),
    y=alt.Y(
        "relative_frequency:Q",
        title="Overall relative frequency",
        scale=alt.Scale(type="log"),
    ),
    color=alt.Color(
        "register:N",
        scale=alt.Scale(domain=["Speech", "Narrative"], range=["tomato", "steelblue"]),
        legend=alt.Legend(title="More common in"),
    ),
    tooltip=["lemma:N", "log_likelihood:Q", "relative_frequency:Q"],
).properties(
    title="Speech vs. Narrative: lemma distinctiveness vs. overall frequency",
    width=650,
    height=450,
)

alt.Chart(...)

In [40]:
narrative_series, speech_series = narrative.align(speech, fill_value=0)

total_narrative = narrative_series.sum()
total_speech = speech_series.sum()

# apply Laplace smoothing
V = len(narrative_series)
alpha = 1
p_narrative = (narrative_series + alpha) / (total_narrative + alpha * V)
p_speech = (speech_series + alpha) / (total_speech + alpha * V)
log_ratio = np.log(p_speech / p_narrative)

# Dunning G² + p-value on the raw (unsmoothed) counts, for significance alongside log_ratio
g_squared = log_likelihood(speech_series, narrative_series, total_speech, total_narrative)
p_value = chi2.sf(g_squared.abs(), df=1)
_, p_value_fdr, _, _ = multipletests(p_value, method="fdr_bh")

log_ratio_df = pd.DataFrame({
    "lemma": log_ratio.index,
    "log_ratio": log_ratio.values,
    "g_squared": g_squared.values,
    "p_value": p_value,
    "p_value_fdr": p_value_fdr,
})

/Users/pletcher/code/writing/articles/2026-06-13_ccc-2026/.venv/lib/python3.13/site-packages/pandas/core/arraylike.py:402: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)


In [41]:
log_ratio_df.sort_values("log_ratio").to_csv("../csv/speech_vs_narrative-log-ratio.csv", index=False)

Parry 1956, Griffin 1986, Worman 2002, McConnell 2025 — what can I actually add to this conversation?

- How strongly each lemma is associated with each register
- Collocates?

# Proportions

In [ ]:
narrative_df = pd.read_parquet("../parquet/homeric_narrative.parquet")
speech_df = pd.read_parquet("../parquet/homeric_speeches.parquet")

In [ ]:
narrative_line_counts = narrative_df.groupby("title").size().rename("narrative_lines")

speech_line_counts = (
    speech_df["content"]
    .str.count("\n")
    .add(1)
    .groupby(speech_df["title"])
    .sum()
    .rename("speech_lines")
)

line_counts = pd.concat([narrative_line_counts, speech_line_counts], axis=1)
line_counts["total_lines"] = line_counts["narrative_lines"] + line_counts["speech_lines"]
line_counts["pct_speech"] = line_counts["speech_lines"] / line_counts["total_lines"]
line_counts["pct_narrative"] = line_counts["narrative_lines"] / line_counts["total_lines"]

line_counts

,narrative_lines,speech_lines,total_lines,pct_speech,pct_narrative
title,,,,,
Iliad,8633,7125,15758,0.452151,0.547849
Odyssey,3878,8404,12282,0.684253,0.315747


In [ ]:
old_homer = pd.read_parquet("../parquet/old_homer.parquet")

In [ ]:
old_homer.groupby("title").size().rename("total_lines")

title
Iliad      15687
Odyssey    12107
Name: total_lines, dtype: int64

# Bigrams

In [59]:
bigram_df = pd.read_parquet("../parquet/bigrams_by_epic_and_register.parquet")

bigram_df.odyssey.speech.sort_values()

bigram
[̓ἀλκίνοος, δῶμα]       0
[φύλλον, φλοιός]        0
[φύλλον, χύσις]         0
[φύλλον, ἄνεμος]        0
[φύλλον, ἔνειμι]        0
                       ..
[Ζεύς, πατήρ]          23
[ἀμείβω, προσεῖπον]    26
[ναῦς, θοός]           29
[ναῦς, μέλας]          31
[πατρίς, γαῖα]         57
Name: speech, Length: 60321, dtype: int64